# 02 · Classification

Full classification pipeline with all seven enhancement modules:

- **Confidence scoring** — flags low-confidence records for human review
- **Feedback injection** — uses stored human corrections as few-shot examples
- **Hierarchical classification** — optional L1→L2 two-step approach
- **Incremental processing** — skips records already classified (daily batches)
- **Cost tracking** — per-module token usage and USD estimates
- **Language detection** — detects and optionally translates non-English input
- **Quality dashboard** — automated quality report with run-over-run comparison

**Workflow:**
```
SAMPLE_N = 50   → ~2-3 min sanity check
SAMPLE_N = 500  → ~15 min quality review
SAMPLE_N = None → full run (use rca run for production)
```

### Policy taxonomy harmonisation — CAR Manual v3.37 + Content Coverage Guide (June 2026)

**New policy codes added:**

| Code | Label | Source |
|---|---|---|
| PA10 | Article-in-Press (AiP) eligibility and processing | CAR Manual §5 |
| PA11 | Excluded item types (full CAR §1 exclusion list) | CAR Manual §1 |
| PU2.x | Customer root cause | Merged from policy_understanding into policy_compliance |
| CR1.8 | AiP visibility or missing (contact reason) | CAR Manual §5 |

**Modules changed:**

- `policy_compliance`: expanded with PA10/PA11 + customer understanding fields (`customer_understanding_issue`, `customer_root_cause_code`)
- `policy_understanding`: disabled (fields merged into policy_compliance)

After updating the Excel, run Section 2 (force-rebuild) to regenerate `policy_compliance.json` and `policy_compliance.py`.  
Use `rca build build --force policy_compliance` to regenerate only the compliance module.


### PA12–PA17 additions (June 2026) — first classification run required

**New policy codes added to `policy_compliance` module:**

| Code | Label | Source |
|---|---|---|
| PA12 | Patent indexing and coverage | Content Coverage Guide §2.3 |
| PA13 | Author identity and profile management | Content Coverage Guide §3.3 |
| PA14 | Affiliation profile curation | Content Coverage Guide §3.4 |
| PA15 | Open Access classification | Content Coverage Guide §3.1 |
| PA16 | Errata and record corrections | CCG §3.1 + CAR Manual §6 |
| PA17 | Cited references coverage and linkage | Content Coverage Guide §3.2, §4.5 |

After updating the Excel, rebuild the module:
`rca build build --force policy_compliance`

Then re-run classification (notebook 12) to produce a new `Scopus_Content_Policy_enriched.xlsx` with PA12–PA17 tickets included. The R analysis script (`scopus_effort_master_v1.R`) will automatically handle the new codes — `policy_doc_map` already contains entries for PA12–PA17.


## 0 · Imports

In [ ]:
WORKSPACE = "scopus_content"  # folder under use_cases/ to work in — change to run on another use case
import sys, os
from pathlib import Path

# Ensure project root is on sys.path and is the working directory
_root = Path().absolute()
# Navigate up until we find the rca_pipeline package
while not (_root / "use_cases").is_dir() and _root.parent != _root:
    _root = _root.parent
_root = _root / "use_cases" / WORKSPACE  # the use-case folder this notebook works in
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"Working directory: {os.getcwd()}")


In [ ]:
import asyncio, json, time
from dataclasses import replace as _replace
from pathlib import Path
import nest_asyncio
nest_asyncio.apply()
import pandas as pd
import logging

logging.basicConfig(level=logging.INFO,
    format="%(asctime)s │ %(levelname)-7s │ %(name)-20s │ %(message)s",
    datefmt="%H:%M:%S")
logger = logging.getLogger("nb_classify")

from rca_pipeline.config import RunConfig, get_credentials
from rca_pipeline.taxonomy import load_taxonomies
from rca_pipeline.utils import read_input_file, drop_enrichment_cols, post_process, write_output_file
from rca_pipeline.pipeline import prepare_batched_records, run_classification_module, MODULE_REGISTRY
from rca_pipeline.llm_client import (
    set_system_prompt_suffix, clear_system_prompt_suffix,
    register_cost_tracker, set_cost_tracker_module, unregister_cost_tracker,
)

# Enhancement imports (graceful — missing modules don't crash the notebook)
try:
    from rca_pipeline.enhancements.confidence import enrich_with_confidence, flag_low_confidence, export_review_queue, inject_confidence_instruction
    _HAS_CONFIDENCE = True
except ImportError:
    _HAS_CONFIDENCE = False

try:
    from rca_pipeline.enhancements.feedback import FeedbackStore
    _HAS_FEEDBACK = True
except ImportError:
    _HAS_FEEDBACK = False

try:
    from rca_pipeline.enhancements.hierarchical import run_hierarchical_module
    _HAS_HIERARCHICAL = True
except ImportError:
    _HAS_HIERARCHICAL = False

try:
    from rca_pipeline.enhancements.incremental import IncrementalRunner
    _HAS_INCREMENTAL = True
except ImportError:
    _HAS_INCREMENTAL = False

try:
    from rca_pipeline.enhancements.cost_tracker import CostTracker
    _HAS_COST = True
except ImportError:
    _HAS_COST = False

try:
    from rca_pipeline.enhancements.language import detect_language, preprocess_multilingual
    _HAS_LANGUAGE = True
except ImportError:
    _HAS_LANGUAGE = False

try:
    from rca_pipeline.enhancements.quality_dashboard import QualityDashboard, run_quality_check
    _HAS_QUALITY = True
except ImportError:
    _HAS_QUALITY = False

print(f"Registered modules: {list(MODULE_REGISTRY.keys())}")
print(f"\nEnhancements available:")
for name, available in [
    ("Confidence scoring", _HAS_CONFIDENCE),
    ("Feedback (few-shot)", _HAS_FEEDBACK),
    ("Hierarchical (L1→L2)", _HAS_HIERARCHICAL),
    ("Incremental processing", _HAS_INCREMENTAL),
    ("Cost tracking", _HAS_COST),
    ("Language detection", _HAS_LANGUAGE),
    ("Quality dashboard", _HAS_QUALITY),
]:
    print(f"  {'✓' if available else '✗'} {name}")

## 1 · Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
INPUT_FILE   = "data\Scopus_Content_Collapsed.xlsx"
OUTPUT_FILE  = "runs/output/Scopus_Content_Policy_enriched.xlsx"
TAXONOMY_DIR = "taxonomies_scopus_content"

# ── Crash recovery ────────────────────────────────────────────────────────
CHECKPOINT_FILE = "data/checkpoint_results.jsonl"
RESUME          = False     # False = delete checkpoint and start fresh

# ── Sampling ──────────────────────────────────────────────────────────────
SAMPLE_N     = None           # ← START HERE; None for full run
SAMPLE_SEED  = 42
STRATIFY_COL = None

# ── Column names ──────────────────────────────────────────────────────────
ID_COL   = "Reference #"
TEXT_COL = "Conversation"

# ── Modules ───────────────────────────────────────────────────────────────
MODULES = ["policy_compliance"] # list(MODULE_REGISTRY.keys())
print("Modules:", MODULES)

# ── Batching & LLM ────────────────────────────────────────────────────────
BATCH_SIZE          = 12
MAX_CHARS_PER_BATCH = 60_000
MODEL               = "azure/gpt-5.4-mini"
TEMPERATURE         = 0.0
MAX_TOKENS          = 8192
TIMEOUT             = 120
CONCURRENCY         = 3
RETRIES             = 4

### Enhancement settings

In [ ]:
# ── Enhancement 1: Confidence scoring ─────────────────────────────────────
ENABLE_CONFIDENCE    = True     # Add confidence scores to each classification
CONFIDENCE_THRESHOLD = 0.50     # Flag records below this for human review
EXPORT_REVIEW_QUEUE  = True     # Write low-confidence records to separate file

# ── Enhancement 2: Feedback (few-shot injection) ─────────────────────────
FEEDBACK_DIR         = "runs/feedback"
FEEDBACK_MAX_EXAMPLES = 5
FEEDBACK_STRATEGY    = "smart"  # "recent", "diverse", or "frequent"

# ── Enhancement 3: Hierarchical (two-step L1→L2) ────────────────────────
HIERARCHICAL_MODULES = []       # e.g. ["contact_reason", "customer_effort"] to enable for specific modules

# ── Enhancement 4: Incremental processing ────────────────────────────────
INCREMENTAL          = False    # Only classify new/changed records
INCREMENTAL_STATE    = "data/incremental_state.jsonl"

# ── Enhancement 5: Cost tracking ─────────────────────────────────────────
TRACK_COSTS          = True
COST_REPORT_DIR      = "runs/cost_reports"

# ── Enhancement 6: Language detection ────────────────────────────────────
DETECT_LANGUAGE      = True    # Detect language of each transcript
LANGUAGE_STRATEGY    = "translate"  # "translate", "prompt", or "detect"

# ── Enhancement 7: Quality dashboard ─────────────────────────────────────
RUN_QUALITY_CHECK    = True
QUALITY_DIR          = "runs/output"
PREVIOUS_REPORT      = None     # Path to previous quality_report.json for comparison

In [ ]:
cfg = RunConfig(
    input_file=INPUT_FILE, output_file=OUTPUT_FILE, taxonomy_dir=TAXONOMY_DIR,
    id_col=ID_COL, text_col=TEXT_COL, mode="classify", modules=MODULES,
    batch_size=BATCH_SIZE, max_chars_per_batch=MAX_CHARS_PER_BATCH,
    model=MODEL, temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
    timeout=TIMEOUT, concurrency=CONCURRENCY, retries=RETRIES,
    # Enhancement config
    enable_confidence=ENABLE_CONFIDENCE,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    feedback_dir=FEEDBACK_DIR,
    feedback_max_examples=FEEDBACK_MAX_EXAMPLES,
    feedback_strategy=FEEDBACK_STRATEGY,
    hierarchical_modules=HIERARCHICAL_MODULES,
    incremental=INCREMENTAL,
    incremental_state_file=INCREMENTAL_STATE,
    track_costs=TRACK_COSTS,
    cost_report_dir=COST_REPORT_DIR,
    detect_language=DETECT_LANGUAGE,
    language_strategy=LANGUAGE_STRATEGY,
    run_quality_check=RUN_QUALITY_CHECK,
    quality_report_dir=QUALITY_DIR,
    previous_quality_report=PREVIOUS_REPORT,
)

creds   = get_credentials()
missing = [k for k, v in creds.items() if not v]
print("Missing credentials:", missing if missing else "None ✓")
print(f"Sample N: {SAMPLE_N if SAMPLE_N else 'FULL DATASET'}")

## 2 · Load taxonomies & data

In [ ]:
taxonomies = load_taxonomies(cfg.taxonomy_dir)
loaded = [k for k, v in taxonomies.items() if v]
empty  = [k for k, v in taxonomies.items() if not v]
print(f"Loaded:  {loaded}")
if empty: print(f"Missing (optional/legacy): {empty}")

In [ ]:
df_full = read_input_file(cfg.input_file)
df_full = drop_enrichment_cols(df_full)
print(f"Loaded {len(df_full):,} rows | Columns: {list(df_full.columns)}")
df_full.head(3)

## 2a · Language detection *(Enhancement 6)*

In [ ]:
# Persist language detection to output — strip underscore prefix so post_process keeps it
# (post_process() strips all columns with leading underscore; Block 8H checks 'detected_language')
if "_detected_language" in df_full.columns:
    df_full = df_full.rename(columns={
        "_detected_language": "detected_language",
        "_language_confidence": "language_confidence",
    })
    print(f"Renamed: _detected_language → detected_language  ({df_full['detected_language'].notna().sum():,} rows with language data)")


In [ ]:
if DETECT_LANGUAGE and _HAS_LANGUAGE:
    print("Running language detection...")
    if LANGUAGE_STRATEGY == "translate":
        # Translation requires async — use detect-only here, translate in classification
        df_full = detect_language(df_full, text_col=TEXT_COL)
    else:
        df_full = detect_language(df_full, text_col=TEXT_COL)

    # Show distribution
    lang_col = "_detected_language"
    if lang_col in df_full.columns:
        print("\nLanguage distribution:")
        for lang, cnt in df_full[lang_col].value_counts().items():
            pct = cnt / len(df_full) * 100
            print(f"  {lang:5s}  {cnt:6,}  ({pct:5.1f}%)")
else:
    if DETECT_LANGUAGE and not _HAS_LANGUAGE:
        print("⚠️  language.py not available — skipping language detection")
    else:
        print("Language detection disabled (set DETECT_LANGUAGE = True to enable)")

## 2b · Feedback preview *(Enhancement 2)*

In [ ]:
if _HAS_FEEDBACK:
    store = FeedbackStore(FEEDBACK_DIR)
    stats = store.get_stats()

    if stats:
        print("Few-shot corrections available:")
        for module, count in sorted(stats.items()):
            print(f"  {module:25s}  {count} corrections")
            preview = store.build_few_shot_prompt(module, max_examples=2, max_text_chars=200)
            if preview:
                lines = preview.strip().split("\n")
                for line in lines[:6]:
                    print(f"    {line[:100]}")
                if len(lines) > 6:
                    print(f"    ... ({len(lines)-6} more lines)")
    else:
        print("No feedback corrections available yet.")
        print("After reviewing results, use FeedbackStore to add corrections.")
else:
    print("Feedback module not available")

## 3 · Sample selection

In [ ]:
if SAMPLE_N is not None:
    n = min(SAMPLE_N, len(df_full))
    if STRATIFY_COL and STRATIFY_COL in df_full.columns:
        groups = df_full[STRATIFY_COL].value_counts(normalize=True)
        frames = [
            grp.sample(n=min(max(1, round(frac * n)), len(grp)), random_state=SAMPLE_SEED)
            for val, frac in groups.items()
            for grp in [df_full[df_full[STRATIFY_COL] == val]]
        ]
        df = pd.concat(frames).sample(frac=1, random_state=SAMPLE_SEED).iloc[:n].reset_index(drop=True)
        print(f"Stratified sample on '{STRATIFY_COL}': {len(df):,} records")
    else:
        df = df_full.sample(n=n, random_state=SAMPLE_SEED).reset_index(drop=True)
        print(f"Random sample: {len(df):,} records (seed={SAMPLE_SEED})")
    _out = Path(cfg.output_file)
    cfg  = _replace(cfg, output_file=str(_out.parent / f"{_out.stem}_sample{n}{_out.suffix}"))
    print(f"Output: {cfg.output_file}")
else:
    df = df_full
    print(f"Full dataset: {len(df):,} records | Output: {cfg.output_file}")

# Char-length stats
char_len = df[TEXT_COL].astype(str).str.len()
print(f"\nChar lengths: mean={char_len.mean():.0f}  p75={char_len.quantile(.75):.0f}  max={char_len.max():.0f}")
est = -(-len(df) // cfg.batch_size)
print(f"Est. batches: {est:,}  |  Est. API calls: {est * len(cfg.modules):,}")

# Cost estimate (Enhancement 5)
if TRACK_COSTS and _HAS_COST:
    tracker = CostTracker(model=MODEL)
    estimate = tracker.estimate_run_cost(len(df), modules=MODULES)
    print(f"\nEstimated cost: ${estimate['estimated_total_usd']:.4f}")
    for mod, cost in estimate.get("by_module", {}).items():
        print(f"  {mod:25s}  ~${cost:.4f}")

## 4 · Batches & checkpoint

In [ ]:
batches = prepare_batched_records(df, cfg)
sizes   = [len(b) for b in batches]
print(f"Batches: {len(batches)} | sizes min={min(sizes)} max={max(sizes)} mean={sum(sizes)/len(sizes):.1f}")
assert sum(sizes) == len(df), "Batch record count mismatch"
print("Record count check ✓")

# ── Checkpoint ────────────────────────────────────────────────────────────
_use_ckpt = CHECKPOINT_FILE is not None and SAMPLE_N is None
ckpt_path = Path(CHECKPOINT_FILE) if _use_ckpt else None

ckpt_results: list = []

_MODULE_SENTINEL = {
    "contact_reason":  "contact_reason_primary_code",
    # "root_cause":      "root_cause_primary_code",
    # "resolution":      "resolution_code",
    # "escalation":      "escalated",
    # "troubleshooting": "troubleshooting_step_codes",
    # "metadata":        "short_summary",
    "customer_effort": "effort_primary_code",
}

completed_uids_by_module: dict = {m: set() for m in cfg.modules}

if _use_ckpt and RESUME and ckpt_path.exists():
    with open(ckpt_path) as _f:
        for line in _f:
            if line.strip():
                r = json.loads(line)
                ckpt_results.append(r)
                uid = str(r.get("uid", ""))
                for mod in cfg.modules:
                    sentinel = _MODULE_SENTINEL.get(mod, "")
                    if not sentinel or sentinel not in r:
                        continue
                    val = r[sentinel]
                    if isinstance(val, list):
                        completed_uids_by_module[mod].add(uid)
                    elif val not in (None, "", False):
                        completed_uids_by_module[mod].add(uid)
                    elif mod == "escalation" and isinstance(val, bool):
                        completed_uids_by_module[mod].add(uid)

    print(f"\n♻️  Resuming from checkpoint:")
    for mod in cfg.modules:
        n    = len(completed_uids_by_module[mod])
        pct  = n / len(df) * 100
        bar  = "█" * int(pct / 2.5)
        print(f"   {mod:25s}  {n:6,}/{len(df):,}  ({pct:5.1f}%)  {bar}")

elif _use_ckpt:
    if ckpt_path.exists() and not RESUME:
        ckpt_path.unlink()
        print("🗑️  RESUME=False — deleted checkpoint, starting fresh")
    else:
        print(f"📝 Fresh start — results will save to: {ckpt_path}")
else:
    print("ℹ️  Checkpointing disabled for sample run")

print()
total_remaining = sum(
    sum(1 for b in batches if any(str(r.get("uid","")) not in completed_uids_by_module.get(m, set()) for r in b))
    for m in cfg.modules
)
print(f"Batch-module pairs remaining: {total_remaining:,}")
for mod in cfg.modules:
    done_set = completed_uids_by_module.get(mod, set())
    todo     = sum(1 for b in batches if any(str(r.get("uid","")) not in done_set for r in b))
    skip     = len(batches) - todo
    print(f"   {mod:25s}  skip={skip:4d}  todo={todo:4d}")

## 5 · Run classification with live progress

In [ ]:
# Initialise cost tracker (Enhancement 5)
cost_tracker = CostTracker(model=MODEL) if TRACK_COSTS and _HAS_COST else None

# B2 fix: register cost tracker so _llm_call records usage
if cost_tracker is not None:
    register_cost_tracker(cost_tracker)

# B3 fix: pre-build per-module feedback prompts
_feedback_prompts = {}
if _HAS_FEEDBACK:
    _fb_store = FeedbackStore(FEEDBACK_DIR)
    for _mod in cfg.modules:
        _fp = _fb_store.build_few_shot_prompt(_mod, max_examples=FEEDBACK_MAX_EXAMPLES, selection_strategy=FEEDBACK_STRATEGY)
        if _fp:
            _feedback_prompts[_mod] = _fp

# B1 fix: build confidence instruction
_confidence_instr = ""
if ENABLE_CONFIDENCE and _HAS_CONFIDENCE:
    _confidence_instr = inject_confidence_instruction("")

async def run_with_progress():
    def _batches_for_module(mod: str) -> list:
        done = completed_uids_by_module.get(mod, set())
        if not done:
            return batches
        return [b for b in batches if any(str(r.get("uid","")) not in done for r in b)]

    semaphore       = asyncio.Semaphore(cfg.concurrency)
    module_todo     = {m: len(_batches_for_module(m)) for m in cfg.modules}
    done            = {m: 0 for m in cfg.modules}
    recs            = {m: 0 for m in cfg.modules}
    errs            = {m: 0 for m in cfg.modules}
    t_start         = {m: None for m in cfg.modules}
    all_new         = []

    # Check which modules use hierarchical classification
    hierarchical_mods = set(HIERARCHICAL_MODULES) if _HAS_HIERARCHICAL else set()

    def _table(first=False):
        n = len(cfg.modules)
        if not first:
            print(f"\033[{n+2}A", end="")
        now = time.time()
        print(f"{'Module':<22} {'Done':>6}/{'Todo':<6} {'Recs':>8} {'Err':>5}  Status")
        print("─" * 65)
        for m in cfg.modules:
            todo = module_todo[m]
            h = " (L1→L2)" if m in hierarchical_mods else ""
            if t_start[m] is None:
                st = "waiting" + h if todo > 0 else "✓ nothing to do"
            elif done[m] >= todo:
                elapsed = now - t_start[m]
                st = f"✓ done ({elapsed:.0f}s)" + h
            else:
                pct     = done[m] / todo * 100 if todo else 100
                elapsed = now - t_start[m]
                st = f"{pct:5.1f}% ({elapsed:.0f}s)" + h
            es = f"⚠{errs[m]}" if errs[m] else ""
            print(f"  {m:<20} {done[m]:>6}/{todo:<6} {recs[m]:>8} {es:>5}  {st}")
        print("", flush=True)

    async def _one(batch, mod):
        if t_start[mod] is None:
            t_start[mod] = time.time()
        async with semaphore:
            try:
                # B1/B3 fix: set per-module prompt suffix
                suffix_parts = []
                if _confidence_instr:
                    suffix_parts.append(_confidence_instr)
                if mod in _feedback_prompts:
                    suffix_parts.append(_feedback_prompts[mod])
                set_system_prompt_suffix("\n".join(suffix_parts))
                set_cost_tracker_module(mod)

                if mod in hierarchical_mods:
                    result = await run_hierarchical_module(mod, [batch], taxonomies, cfg)
                else:
                    result = await run_classification_module(mod, [batch], taxonomies, cfg)

                # Enhancement 1: Enrich with confidence
                if ENABLE_CONFIDENCE and _HAS_CONFIDENCE:
                    result = enrich_with_confidence(result, threshold=CONFIDENCE_THRESHOLD)

                return mod, result, None
            except Exception as e:
                return mod, [], e

    coros = [_one(b, m) for m in cfg.modules for b in _batches_for_module(m)]
    total_calls = len(coros)
    print(f"\n{len(cfg.modules)} modules, {len(coros)} batch-calls remaining (concurrency={cfg.concurrency})\n")
    _table(first=True)

    cp = open(ckpt_path, "a") if _use_ckpt else None
    try:
        for coro in asyncio.as_completed(coros):
            try:
                mod, result, err = await coro
            except Exception as e:
                logger.error("Unhandled batch error: %s", e)
                _table()
                continue
            done[mod] += 1
            if err is not None:
                errs[mod] += 1
                logger.error("%s batch error: %s", mod, err)
            else:
                recs[mod] += len(result)
                all_new.extend(result)
                if cp:
                    for rec in result:
                        cp.write(json.dumps(rec) + "\n")
                    cp.flush()
            _table()
    finally:
        if cp: cp.close()

    return all_new

start       = time.time()
new_results = await run_with_progress()
clear_system_prompt_suffix()
if cost_tracker is not None:
    unregister_cost_tracker()
all_results = ckpt_results + new_results
elapsed     = time.time() - start

print(f"Done — {len(new_results):,} new + {len(ckpt_results):,} checkpoint = {len(all_results):,} total  ({elapsed:.1f}s)")
if SAMPLE_N and elapsed > 0:
    rate = len(df) / elapsed
    print(f"Extrapolated full run: ~{len(df_full)/rate/60:.0f} min  ({rate:.1f} rec/s across {len(cfg.modules)} modules)")

## 6 · Merge by UID

In [ ]:
grouped = {}
for r in all_results:
    uid = str(r.get("uid",""))
    grouped.setdefault(uid, {"uid": uid}).update(r)

merged = list(grouped.values())
print(f"Merged: {len(merged):,} unique UIDs (expected {len(df):,})")
errors = [r for r in merged if "_error" in r]
if errors:
    print(f"\n⚠️  {len(errors)} error records:")
    for e in errors[:5]: print(f"  uid={e.get('uid')}: {e.get('_error')}")
else:
    print("✓ No errors")

## 7 · Post-processing

In [ ]:
df_enriched = post_process(df, merged, taxonomies, cfg)

# Enhancement 1: Flag low-confidence rows
if ENABLE_CONFIDENCE and _HAS_CONFIDENCE and "_confidence" in df_enriched.columns:
    df_enriched = flag_low_confidence(df_enriched, threshold=CONFIDENCE_THRESHOLD)

print(f"Enriched: {len(df_enriched):,} rows × {len(df_enriched.columns)} columns")
print("\nColumns:")
for c in df_enriched.columns: print(f"  {c}")

## 8 · Quality dashboard *(Enhancement 7)*

In [ ]:
if RUN_QUALITY_CHECK and _HAS_QUALITY:
    previous = None
    if PREVIOUS_REPORT and Path(PREVIOUS_REPORT).exists():
        previous = QualityDashboard.load_previous(PREVIOUS_REPORT)

    dashboard = QualityDashboard(df_enriched, taxonomies, cfg, previous)
    report = dashboard.generate()
    dashboard.print_summary()
else:
    print("Quality dashboard disabled or not available.")
    print("Set RUN_QUALITY_CHECK = True to enable.")

## 8a · Coverage details

In [ ]:
print("Classification coverage (% non-empty primary codes):")
for col in [c for c in df_enriched.columns
            if c.endswith("_primary_code") or c == "contact_reason_primary_code"]:
    filled = df_enriched[col].notna() & (df_enriched[col].astype(str).str.strip() != "")
    pct    = filled.sum() / len(df_enriched) * 100
    print(f"  {col:45s} {filled.sum():5d}/{len(df_enriched)}  {pct:5.1f}%  {'█'*int(pct/2.5)}")

In [ ]:
# Effort score distribution
if "effort_score" in df_enriched.columns:
    print("effort_score_distribution:")
    for val, cnt in df_enriched["effort_score"].value_counts(dropna=False).items():
        pct = cnt / len(df_enriched) * 100
        print(f"  {str(val):20s} {cnt:5d}  {pct:5.1f}%  {'█'*int(pct/2.5)}")

In [ ]:
# Customer effort distribution
if "effort_score" in df_enriched.columns:
    print("Customer effort score:")
    for val, cnt in df_enriched["effort_score"].value_counts(dropna=False).items():
        pct = cnt / len(df_enriched) * 100
        print(f"  {str(val):15s} {cnt:5d}  {pct:5.1f}%  {'█'*int(pct/2.5)}")

## 8b · Confidence analysis *(Enhancement 1)*

In [ ]:
if ENABLE_CONFIDENCE and "_confidence" in df_enriched.columns:
    conf = pd.to_numeric(df_enriched["_confidence"], errors="coerce")
    valid = conf.dropna()

    if len(valid) > 0:
        print(f"Confidence scores: {len(valid):,} records")
        print(f"  Mean:   {valid.mean():.3f}")
        print(f"  Median: {valid.median():.3f}")
        print(f"  Std:    {valid.std():.3f}")
        print(f"  Min:    {valid.min():.3f}")
        print(f"  Max:    {valid.max():.3f}")

        # Distribution
        print(f"\nDistribution:")
        for label, lo, hi in [("Very high (0.9-1.0)", 0.9, 1.01),
                                ("High (0.7-0.9)", 0.7, 0.9),
                                ("Moderate (0.5-0.7)", 0.5, 0.7),
                                ("Low (0.0-0.5)", 0.0, 0.5)]:
            n = ((valid >= lo) & (valid < hi)).sum()
            pct = n / len(valid) * 100
            print(f"  {label:25s}  {n:5d}  ({pct:5.1f}%)")

        # Flagged for review
        if "_needs_review" in df_enriched.columns:
            n_review = df_enriched["_needs_review"].sum()
            print(f"\nFlagged for human review: {n_review:,} ({n_review/len(df_enriched)*100:.1f}%)")
    else:
        print("No valid confidence scores found.")
else:
    print("Confidence scoring disabled (set ENABLE_CONFIDENCE = True)")

## 8c · Top classifications

In [ ]:
# Top contact reasons
if "contact_reason_primary_label" in df_enriched.columns:
    print("Top 15 contact reasons:")
    display(df_enriched["contact_reason_primary_label"]
            .value_counts().head(15)
            .rename_axis("contact_reason").reset_index(name="count"))

# Top customer efforts
if "customer_effort_primary_label" in df_enriched.columns:
    print("\nTop 15 customer efforts:")
    display(df_enriched["customer_effort_primary_label"]
            .value_counts().head(15)
            .rename_axis("customer_effort").reset_index(name="count"))

## 9 · Per-record review

In [ ]:
REVIEW_N = 5

key_cols = [
    ID_COL, TEXT_COL,
    "contact_reason_primary_label",
    "root_cause_primary_label", "root_cause_justification",
    "root_cause_evidence", "root_cause_evidence_type",
    "resolution_status", "escalated",
    "effort_score", "effort_evidence", "repeat_contact",
    "short_summary",
    "_confidence", "_needs_review",
]
review_cols = [c for c in key_cols if c in df_enriched.columns]

# Prioritise low-confidence records for review if confidence is enabled
if "_confidence" in df_enriched.columns:
    review_df = df_enriched.sort_values("_confidence", ascending=True).head(REVIEW_N)
    print(f"Reviewing {REVIEW_N} lowest-confidence records:\n")
else:
    review_df = df_enriched[review_cols].sample(n=min(REVIEW_N, len(df_enriched)), random_state=SAMPLE_SEED)
    print(f"Reviewing {REVIEW_N} random records:\n")

pd.set_option("display.max_colwidth", 160)
for _, row in review_df.iterrows():
    print("="*80)
    for col in review_cols:
        val = row.get(col, "")
        if col == TEXT_COL and isinstance(val, str) and len(val) > 500:
            val = val[:500] + " [...]"
        # Highlight low-confidence and review flags
        prefix = "  "
        if col == "_confidence" and pd.notna(val):
            try:
                v = float(val)
                prefix = "⚠ " if v < CONFIDENCE_THRESHOLD else "  "
            except: pass
        if col == "_needs_review" and val == True:
            prefix = "⚠ "
        print(f"{prefix}{col:40s}: {val}")
print("="*80)

## 10 · Write output

In [ ]:
if _use_ckpt and ckpt_path and ckpt_path.exists():
    sz   = ckpt_path.stat().st_size / 1024
    with open(ckpt_path) as _f:
        n_cp = sum(1 for l in _f if l.strip())
    print(f"Checkpoint: {n_cp:,} records | {sz:.1f} KB | {ckpt_path}")

out_path = write_output_file(df_enriched, cfg.output_file)
print(f"\n✓ {out_path.resolve()}")
print(f"  {len(df_enriched):,} rows × {len(df_enriched.columns)} columns")

# Export review queue (Enhancement 1)
if EXPORT_REVIEW_QUEUE and ENABLE_CONFIDENCE and _HAS_CONFIDENCE:
    review_path = str(Path(cfg.output_file).parent / "review_queue.xlsx")
    export_review_queue(df_enriched, review_path)

# Save quality report (Enhancement 7)
if RUN_QUALITY_CHECK and _HAS_QUALITY and 'dashboard' in dir():
    report_path = Path(QUALITY_DIR) / "quality_report.json"
    dashboard.save(report_path)
    try:
        dashboard.save_excel(Path(QUALITY_DIR) / "quality_report.xlsx")
    except Exception as e:
        print(f"Could not save Excel quality report: {e}")

# Save cost report (Enhancement 5)
if TRACK_COSTS and _HAS_COST and 'cost_tracker' in dir() and cost_tracker is not None:
    cost_tracker.print_summary()
    Path(COST_REPORT_DIR).mkdir(parents=True, exist_ok=True)
    cost_tracker.save(Path(COST_REPORT_DIR) / f"costs_{cost_tracker._run_id}.json")

if SAMPLE_N:
    print(f"\nSample run complete. Set SAMPLE_N = None and re-run from Section 3 for the full dataset.")

## 11 · Add corrections *(Enhancement 2)*

After reviewing classifications, add corrections here for future runs.
These corrections will be injected as few-shot examples in subsequent classifications.

In [ ]:
# ── Example: add a single correction ─────────────────────────────────────
# Uncomment and edit to add corrections:

# if _HAS_FEEDBACK:
#     store = FeedbackStore(FEEDBACK_DIR)
#     store.add_correction(
#         module="contact_reason",
#         uid="42",
#         text="Customer called because they cannot log in after password change...",
#         original={"contact_reason_primary_code": "CR1.2"},
#         corrected={"contact_reason_primary_code": "CR3.1"},
#         reason="This is an account recovery issue, not a login blocked issue"
#     )
#     print("Correction added. It will be used as a few-shot example in the next run.")

# ── Or import corrections from the review queue spreadsheet ──────────────
# if _HAS_FEEDBACK:
#     store = FeedbackStore(FEEDBACK_DIR)
#     count = store.import_from_review_file(
#         "runs/output/review_queue_corrected.xlsx",
#         module="contact_reason",
#         text_col=TEXT_COL,
#         uid_col=ID_COL,
#     )
#     print(f"Imported {count} corrections")

print("Uncomment the cells above to add corrections after reviewing classifications.")
print("Corrections are stored in:", FEEDBACK_DIR)

## 12 · Error analysis

In [ ]:
# Analyse error records
error_records = [r for r in merged if "_error" in r]

if error_records:
    from collections import Counter
    print(f"Total error records: {len(error_records)}")
    print(f"Unique UIDs with errors: {len({r['uid'] for r in error_records})}")
    print()

    error_types = Counter()
    for r in error_records:
        err = str(r.get("_error", ""))
        if "ContentPolicy" in err:
            error_types["content_policy_violation"] += 1
        elif "uid_repair_failed" in err:
            error_types["uid_repair_failed"] += 1
        elif "RateLimit" in err:
            error_types["rate_limit_exhausted"] += 1
        elif "Timeout" in err:
            error_types["timeout"] += 1
        else:
            error_types["other: " + err[:60]] += 1

    for err_type, count in error_types.most_common():
        print(f"  {count:5d}  {err_type}")
else:
    print("✓ No error records")

In [ ]:
import json
tax = json.load(open("taxonomies_scopus_content/customer_effort.json"))
for code, node in tax.items():
    print(f"  {code}: description={'description' in node}  →  {node.get('description', '(none)')[:60]}")